# Hallucination Detection v1 — GPU Benchmark

Runs `attention_analyzer.py`, `hypothesis_test.py`, and `confidence_calibrator.py` on a real transformer (Pythia-160m) using Colab's GPU.

**Outputs:**
- Real AUROC, latency, throughput benchmarks
- Attention heatmap visualisations
- `v1_benchmark_results.json` (download and commit to repo)

**Runtime:** ~5 minutes on T4 GPU

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
%pip install -q torch transformers numpy scipy

import sys
import torch

print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 2. Clone repo and add to path ─────────────────────────────────────────
import os

if not os.path.exists('Natural-Hallucination-Analysis'):
    !git clone https://github.com/A-Kuo/Natural-Hallucination-Analysis.git

os.chdir('Natural-Hallucination-Analysis')
sys.path.insert(0, 'v1')
print("Working directory:", os.getcwd())

In [ ]:
# ── 3. Load model on GPU ───────────────────────────────────────────────────
from attention_analyzer import AttentionAnalyzer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL  = "EleutherAI/pythia-160m"

print(f"Loading {MODEL} on {DEVICE}...")
analyzer = AttentionAnalyzer(
    model_name=MODEL,
    device=DEVICE,
    precision=torch.float16 if DEVICE == "cuda" else torch.float32,
)
print(f"Model loaded — {analyzer.num_layers} layers, {analyzer.num_heads} heads")

In [ ]:
# ── 4. Define probe texts (factual vs. hallucination-prone) ───────────────
#
# We use two categories:
#   FACTUAL  — clear, unambiguous facts the model should know confidently
#   CREATIVE — open-ended generation where models tend to fabricate

FACTUAL_PROMPTS = [
    "The capital of France is",
    "Water freezes at 0 degrees",
    "The speed of light is approximately 300,000",
    "Shakespeare wrote Hamlet in",
    "The human genome contains approximately",
    "Mount Everest is located in",
    "The chemical symbol for gold is",
    "Isaac Newton formulated the laws of",
    "The Earth orbits the Sun in approximately",
    "DNA stands for deoxyribonucleic",
]

HALLUCINATION_PRONE_PROMPTS = [
    "The fictional planet Zorath was discovered in",
    "According to the 1823 Treaty of Westbrook, the nations agreed",
    "The quantum principle of entanglement flux states that",
    "Professor Aldric von Steinberg won the Nobel Prize for",
    "The ancient city of Keldrath was founded by Emperor",
    "The chemical compound trisodium phenolphthalein has a melting point of",
    "The 1904 Larson-Hofstadter Theorem proves that",
    "In the language of the Mrevu tribe, the word for sun is",
    "The river Valdecas flows through the mountains of",
    "The philosopher Caius Meridian wrote in his Tractatus that",
]

print(f"Factual prompts:             {len(FACTUAL_PROMPTS)}")
print(f"Hallucination-prone prompts: {len(HALLUCINATION_PRONE_PROMPTS)}")

In [ ]:
# ── 5. Run attention analysis ──────────────────────────────────────────────
import time
import numpy as np

print("Analysing factual prompts...")
factual_results = analyzer.analyze_batch(FACTUAL_PROMPTS)

print("Analysing hallucination-prone prompts...")
halluc_results = analyzer.analyze_batch(HALLUCINATION_PRONE_PROMPTS)

# Summarise
def summarise(results, label):
    entropies = [r.mean_entropy for r in results]
    kls       = [r.total_kl_divergence for r in results]
    latencies = [r.latency_ms for r in results]
    print(f"\n{label}")
    print(f"  mean_entropy:  {np.mean(entropies):.4f} ± {np.std(entropies):.4f}")
    print(f"  total_kl:      {np.mean(kls):.4f} ± {np.std(kls):.4f}")
    print(f"  latency (ms):  {np.mean(latencies):.2f} p95={np.percentile(latencies, 95):.2f}")
    return entropies, kls

f_ent, f_kl = summarise(factual_results,  "FACTUAL")
h_ent, h_kl = summarise(halluc_results,   "HALLUCINATION-PRONE")

In [ ]:
# ── 6. Run hypothesis test + calibrator on all samples ────────────────────
from hypothesis_test import HypothesisTest
from confidence_calibrator import ConfidenceCalibrator

tester     = HypothesisTest()
calibrator = ConfidenceCalibrator()

# Build labels: 0=factual (reliable), 1=hallucination-prone (unreliable)
all_results = factual_results + halluc_results
true_labels = [0] * len(factual_results) + [1] * len(halluc_results)

test_results = []
raw_scores   = []
decisions    = []

for r in all_results:
    tr = tester.test(r)
    test_results.append(tr)
    raw_scores.append(tr.z_combined)
    cal = calibrator.calibrate(tr)
    decisions.append(cal.decision)

# Normalise raw scores to [0,1] for AUROC
raw_arr = np.array(raw_scores)
norm_scores = (raw_arr - raw_arr.min()) / (raw_arr.ptp() + 1e-12)

print("Test complete.")
print("Decisions:", decisions)

In [ ]:
# ── 7. Compute AUROC and report metrics ────────────────────────────────────
from scipy.stats import mannwhitneyu

# AUROC via Mann-Whitney U (equivalent, no sklearn needed)
y = np.array(true_labels)
pos_scores = norm_scores[y == 1]
neg_scores = norm_scores[y == 0]

stat, _ = mannwhitneyu(pos_scores, neg_scores, alternative='greater')
auroc = stat / (len(pos_scores) * len(neg_scores))

# Per-decision counts
from collections import Counter
dcounts = Counter(decisions)

# Latency
all_latencies = [r.latency_ms for r in all_results]
throughput = 1000.0 / np.mean(all_latencies)  # samples/sec

print(f"\n{'═'*50}")
print(f"  v1 REAL BENCHMARK RESULTS")
print(f"{'═'*50}")
print(f"  AUROC:            {auroc:.4f}")
print(f"  Latency mean:     {np.mean(all_latencies):.3f} ms")
print(f"  Latency p95:      {np.percentile(all_latencies, 95):.3f} ms")
print(f"  Throughput:       {throughput:.0f} samples/sec")
print(f"  Device:           {DEVICE}")
print(f"  Model:            {MODEL}")
print(f"  Decisions:        {dict(dcounts)}")
print(f"{'═'*50}")

In [ ]:
# ── 8. Visualise attention entropy distributions ───────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('v1 — Attention Signals: Factual vs. Hallucination-Prone', fontsize=13)

# Entropy comparison
axes[0].boxplot([f_ent, h_ent], labels=['Factual', 'Halluc-prone'], patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[0].set_title('Mean Attention Entropy')
axes[0].set_ylabel('Shannon Entropy (bits)')

# KL comparison
axes[1].boxplot([f_kl, h_kl], labels=['Factual', 'Halluc-prone'], patch_artist=True,
                boxprops=dict(facecolor='#DD8452', alpha=0.7))
axes[1].set_title('Cross-Layer KL Divergence')
axes[1].set_ylabel('KL Divergence (nats)')

# Z-score distribution
z_factual = [test_results[i].z_combined for i in range(len(factual_results))]
z_halluc  = [test_results[i].z_combined for i in range(len(factual_results), len(all_results))]
axes[2].hist(z_factual,  bins=6, alpha=0.7, label='Factual',        color='#4C72B0')
axes[2].hist(z_halluc,   bins=6, alpha=0.7, label='Halluc-prone',   color='#DD8452')
axes[2].axvline(tester.z_critical, color='red', linestyle='--', label=f'z_critical={tester.z_critical:.2f}')
axes[2].set_title('Z-Score Distribution')
axes[2].set_xlabel('Z_combined')
axes[2].legend()

plt.tight_layout()
plt.savefig('v1_attention_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: v1_attention_signals.png")

In [ ]:
# ── 9. Layer-head entropy heatmap for one factual vs. one hallucination ────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result, title in [
    (axes[0], factual_results[0],  f'Factual:\n"{FACTUAL_PROMPTS[0]}..."'),
    (axes[1], halluc_results[0],   f'Hallucination-prone:\n"{HALLUCINATION_PRONE_PROMPTS[0]}..."'),
]:
    im = ax.imshow(result.layer_head_entropy, aspect='auto', cmap='viridis')
    ax.set_xlabel('Head')
    ax.set_ylabel('Layer')
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, label='Entropy (bits)')

fig.suptitle('Per-Layer, Per-Head Attention Entropy', fontsize=12)
plt.tight_layout()
plt.savefig('v1_entropy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: v1_entropy_heatmap.png")

In [ ]:
# ── 10. Save benchmark results as JSON ────────────────────────────────────
import json
from datetime import datetime

benchmark = {
    "timestamp": datetime.utcnow().isoformat(),
    "model": MODEL,
    "device": DEVICE,
    "num_factual": len(FACTUAL_PROMPTS),
    "num_hallucination_prone": len(HALLUCINATION_PRONE_PROMPTS),
    "metrics": {
        "auroc": round(auroc, 4),
        "latency_mean_ms": round(float(np.mean(all_latencies)), 3),
        "latency_p95_ms": round(float(np.percentile(all_latencies, 95)), 3),
        "throughput_samples_per_sec": round(throughput, 1),
    },
    "signal_separation": {
        "factual_entropy_mean": round(float(np.mean(f_ent)), 4),
        "halluc_entropy_mean": round(float(np.mean(h_ent)), 4),
        "factual_kl_mean": round(float(np.mean(f_kl)), 4),
        "halluc_kl_mean": round(float(np.mean(h_kl)), 4),
    },
    "decisions": dict(dcounts),
    "per_sample": [
        {
            "prompt": (FACTUAL_PROMPTS + HALLUCINATION_PRONE_PROMPTS)[i],
            "true_label": int(true_labels[i]),
            "z_score": round(float(test_results[i].z_combined), 4),
            "decision": decisions[i],
            "mean_entropy": round(all_results[i].mean_entropy, 4),
            "total_kl": round(all_results[i].total_kl_divergence, 4),
            "latency_ms": round(all_results[i].latency_ms, 3),
        }
        for i in range(len(all_results))
    ]
}

with open('v1_benchmark_results.json', 'w') as f:
    json.dump(benchmark, f, indent=2)

print("Saved: v1_benchmark_results.json")
print(json.dumps(benchmark["metrics"], indent=2))

In [ ]:
# ── 11. Download results ───────────────────────────────────────────────────
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('v1_benchmark_results.json')
    files.download('v1_attention_signals.png')
    files.download('v1_entropy_heatmap.png')
    print("Downloads triggered.")
else:
    print("Running locally — files saved to:", os.getcwd())